# Preparação dos dados para modelagem

**Projeto:** RNAP 2026/2 — modelo autorregressivo de linguagem

Este notebook transforma o corpus mestre auditado em documentos JSONL e partições de treino, validação e teste. O corpus mestre e os textos-fonte permanecem intactos. A unidade de divisão é o arquivo/obra identificado no inventário; assim, trechos de um mesmo arquivo nunca são espalhados entre partições.

## Decisões de preparação

1. A entrada é `dados/corpus_machado_autoral_nltk_mec.txt`, em UTF-8.
2. São removidos somente os delimitadores técnicos `INÍCIO DO ITEM` e `FIM DO ITEM` acrescentados pelo notebook anterior. Não removemos cabeçalhos, notas editoriais, índices ou conteúdo literário por heurísticas: esses elementos podem ser difíceis de distinguir automaticamente e a versão mestre deve continuar rastreável. Essa escolha pode ensinar ao modelo elementos editoriais; uma filtragem curada será uma comparação experimental posterior.
3. Aplicamos Unicode NFC, normalizamos finais de linha, retiramos espaços no fim das linhas e limitamos sequências de mais de duas linhas vazias a duas. Palavras, pontuação, acentos e ordem do texto são preservados.
4. A partição é determinística (semente fixa), estratificada por categoria aproximada e agrupada por hash de conteúdo normalizado para que cópias integrais idênticas fiquem juntas. O alvo é 80/10/10 por documentos, não por tokens.
5. O teste mede generalização para arquivos/obras não vistos, não para continuações aleatórias de trechos de uma mesma obra. O split de documentos não prova ausência de trechos repetidos entre obras; produzimos auditorias de duplicação exata e distribuição.

Os JSONL são entradas textuais para a próxima etapa de tokenização; ainda não fixam vocabulário, tamanho de contexto ou arquitetura.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import date
import csv
import hashlib
import json
import random
import re
import unicodedata

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "projetos" / "machado-assis").exists():
    if PROJECT_DIR.name == "machado-assis":
        PROJECT_DIR = PROJECT_DIR.parent.parent
    else:
        raise FileNotFoundError("Execute da raiz do workspace ou da pasta projetos/machado-assis.")
MACHADO_DIR = PROJECT_DIR / "projetos" / "machado-assis"
DATA_DIR = MACHADO_DIR / "dados"
MASTER_PATH = DATA_DIR / "corpus_machado_autoral_nltk_mec.txt"
PREP_DIR = DATA_DIR / "modelagem"
PREP_DIR.mkdir(parents=True, exist_ok=True)
SEED = 20260925
SPLIT_RATIOS = {"train": 0.80, "validation": 0.10, "test": 0.10}
print("Entrada:", MASTER_PATH)
print("Saída:", PREP_DIR)


Entrada: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/corpus_machado_autoral_nltk_mec.txt
Saída: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/modelagem


## 1. Leitura e extração dos documentos

In [2]:
if not MASTER_PATH.exists():
    raise FileNotFoundError(f"Corpus mestre ausente: {MASTER_PATH}")
master = MASTER_PATH.read_text(encoding="utf-8")

# Os delimitadores foram adicionados pelo notebook 01 e não pertencem à obra.
item_pattern = re.compile(
    r"===== INÍCIO DO ITEM: (.*?) \| categoria=(.*?) \| fonte=(.*?) =====\n\n"
    r"(.*?)\n\n===== FIM DO ITEM: .*? =====",
    flags=re.DOTALL,
)
items = []
for match in item_pattern.finditer(master):
    label, category, source_file, raw_text = match.groups()
    text = unicodedata.normalize("NFC", raw_text.replace("\r\n", "\n").replace("\r", "\n"))
    lines = [line.rstrip() for line in text.split("\n")]
    text = re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()
    if not text:
        continue
    items.append({
        "document_id": source_file,
        "source_file": source_file,
        "category": category.strip(),
        "title_or_contents_label": label.strip(),
        "text": text,
    })

if not items:
    raise ValueError("Nenhum item foi extraído; confira o formato dos delimitadores do corpus mestre.")
ids = [x["document_id"] for x in items]
assert len(ids) == len(set(ids)), "Há identificadores repetidos; resolver antes da partição."
print("Documentos extraídos:", len(items))
print("Categorias:", dict(sorted(Counter(x["category"] for x in items).items())))
print("Caracteres após normalização conservadora:", sum(len(x["text"]) for x in items))
print("Exemplo (início):\n", items[0]["text"][:700])


Documentos extraídos: 242
Categorias: {'contos': 137, 'critica': 45, 'cronica': 24, 'miscelanea': 9, 'poesia': 7, 'romance': 10, 'teatro': 10}
Caracteres após normalização conservadora: 13699949
Exemplo (início):
 Conto, Contos Fluminenses, 1870

Contos Fluminenses

Texto-fonte:

Obra Completa, Machado de Assis, vol. II,

Rio de Janeiro: Nova Aguilar, 1994.

Publicado originalmente pela
Editora Garnier, Rio de Janeiro, em 1870.

ÍNDICE

MISS DOLLAR

LUÍS
SOARES

A MULHER DE
PRETO

O
SEGREDO DE AUGUSTA

CONFISSÕES DE UMA VIÚVA MOÇA

LINHA
RETA E LINHA CURVA

FREI
SIMÃO

MISS
DOLLAR

ÍNDICE

Capítulo Primeiro

Capítulo II

Capítulo iii

Capítulo iv

Capítulo v

Capítulo vI

Capítulo vII

CAPÍTULO VIII

CAPÍTULO PRIMEIRO

Era conveniente ao romance que o leitor
ficasse muito tempo sem saber quem era Miss Dollar. Mas por outro lado,
sem a apresentação de Miss Dollar, seria o autor obrigado a longas
digres


## 2. Auditoria e agrupamento de duplicatas integrais

Agrupamos conteúdos que sejam iguais após NFC, casefold e remoção de espaços. Esse fingerprint serve apenas para impedir que duplicatas integrais atravessem divisões; ele não altera o texto salvo. A auditoria registra eventuais grupos.


In [3]:
def exact_fingerprint(text):
    canonical = unicodedata.normalize("NFC", text).casefold()
    canonical = re.sub(r"\s+", " ", canonical).strip()
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

fingerprints = defaultdict(list)
for item in items:
    item["content_sha256"] = hashlib.sha256(item["text"].encode("utf-8")).hexdigest()
    item["duplicate_group"] = exact_fingerprint(item["text"])
    fingerprints[item["duplicate_group"]].append(item["document_id"])
duplicate_groups = {k: v for k, v in fingerprints.items() if len(v) > 1}
print("Grupos de documentos integralmente iguais após normalização de espaços:", len(duplicate_groups))
for group, docs in list(duplicate_groups.items())[:20]:
    print(" •", " | ".join(docs))


Grupos de documentos integralmente iguais após normalização de espaços: 0


## 3. Partição reprodutível por obra/arquivo

In [4]:
# Cada duplicate_group é uma unidade indivisível. Balanceamos o volume de
# caracteres dentro de cada categoria para reduzir desvio causado por obras longas.
by_category = defaultdict(list)
item_by_group = {}
for group, docs in fingerprints.items():
    representative = next(x for x in items if x["duplicate_group"] == group)
    by_category[representative["category"]].append(group)
    item_by_group[group] = representative

rng = random.Random(SEED)
group_to_split = {}
allocation_rows = []
for category in sorted(by_category):
    groups = list(by_category[category])
    rng.shuffle(groups)
    groups.sort(key=lambda group: len(item_by_group[group]["text"]), reverse=True)
    n = len(groups)
    category_chars = sum(len(item_by_group[g]["text"]) for g in groups)
    targets = {k: SPLIT_RATIOS[k] * category_chars for k in SPLIT_RATIOS}
    loads = {k: 0 for k in SPLIT_RATIOS}
    counts = {k: 0 for k in SPLIT_RATIOS}
    for index, group in enumerate(groups):
        size = len(item_by_group[group]["text"])
        remaining = n - index
        empty_splits = [k for k in ("train", "validation", "test") if counts[k] == 0]
        if n >= 3 and len(empty_splits) >= remaining:
            candidates = empty_splits
        elif n == 2 and index == 0:
            candidates = ["train", "test"]
        elif n == 1:
            candidates = ["train"]
        else:
            candidates = list(SPLIT_RATIOS)
        split = min(candidates, key=lambda k: (loads[k] + size) / max(targets[k], 1))
        group_to_split[group] = split
        loads[split] += size
        counts[split] += 1
    allocation_rows.append({
        "category": category,
        "unique_groups": n,
        "train_groups": counts["train"],
        "validation_groups": counts["validation"],
        "test_groups": counts["test"],
        "train_characters": loads["train"],
        "validation_characters": loads["validation"],
        "test_characters": loads["test"],
    })

for item in items:
    item["split"] = group_to_split[item["duplicate_group"]]

# Disjunção de IDs e grupos; nenhuma obra nem duplicata integral atravessa splits.
split_ids = {name: {x["document_id"] for x in items if x["split"] == name}
             for name in SPLIT_RATIOS}
split_groups = {name: {x["duplicate_group"] for x in items if x["split"] == name}
                 for name in SPLIT_RATIOS}
for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert not (split_ids[a] & split_ids[b])
    assert not (split_groups[a] & split_groups[b])
assert set.union(*split_ids.values()) == set(ids)
print("Documentos por split:", {k: len(v) for k, v in split_ids.items()})
print("Caracteres por split:", {k: sum(len(x["text"]) for x in items if x["split"] == k) for k in SPLIT_RATIOS})
print("Alocação por categoria:")
for row in allocation_rows:
    print(row)


Documentos por split: {'train': 181, 'validation': 30, 'test': 31}
Caracteres por split: {'train': 11113713, 'validation': 1340673, 'test': 1245563}
Alocação por categoria:
{'category': 'contos', 'unique_groups': 137, 'train_groups': 107, 'validation_groups': 15, 'test_groups': 15, 'train_characters': 4289721, 'validation_characters': 535206, 'test_characters': 534398}
{'category': 'critica', 'unique_groups': 45, 'train_groups': 33, 'validation_groups': 6, 'test_groups': 6, 'train_characters': 780527, 'validation_characters': 96639, 'test_characters': 96548}
{'category': 'cronica', 'unique_groups': 24, 'train_groups': 14, 'validation_groups': 5, 'test_groups': 5, 'train_characters': 2512512, 'validation_characters': 314008, 'test_characters': 313605}
{'category': 'miscelanea', 'unique_groups': 9, 'train_groups': 6, 'validation_groups': 1, 'test_groups': 2, 'train_characters': 41165, 'validation_characters': 4158, 'test_characters': 5056}
{'category': 'poesia', 'unique_groups': 7, 'trai

## 4. Gravação dos dados e manifesto

Cada linha dos arquivos JSONL corresponde a uma obra/arquivo. `document_id` e `source_file` mantêm rastreabilidade, e `text` contém o texto preparado. Para treino causal, a etapa seguinte poderá tokenizar cada texto e formar janelas, preservando os limites documentais.


In [5]:
def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8", newline="\n") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

for split in SPLIT_RATIOS:
    rows = [
        {k: item[k] for k in ("document_id", "source_file", "category", "title_or_contents_label", "content_sha256", "text")}
        for item in items if item["split"] == split
    ]
    write_jsonl(PREP_DIR / f"{split}.jsonl", rows)

with (PREP_DIR / "auditoria_particoes.csv").open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(allocation_rows[0].keys()))
    writer.writeheader()
    writer.writerows(allocation_rows)

item_audit = [{
    "document_id": x["document_id"],
    "category": x["category"],
    "split": x["split"],
    "characters": len(x["text"]),
    "words_approx": len(x["text"].split()),
    "content_sha256": x["content_sha256"],
    "duplicate_group": x["duplicate_group"],
} for x in items]
with (PREP_DIR / "auditoria_documentos.csv").open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(item_audit[0].keys()))
    writer.writeheader()
    writer.writerows(item_audit)

manifest = {
    "project": "RNAP 2026/2 — modelo autorregressivo de linguagem sobre Machado de Assis",
    "created_date": date.today().isoformat(),
    "input_file": str(MASTER_PATH.relative_to(MACHADO_DIR)),
    "input_sha256": hashlib.sha256(MASTER_PATH.read_bytes()).hexdigest(),
    "output_directory": str(PREP_DIR.relative_to(MACHADO_DIR)),
    "document_count": len(items),
    "split_unit": "obra/arquivo (document_id); grupos de conteúdo integralmente duplicado permanecem juntos",
    "seed": SEED,
    "target_split_ratios": SPLIT_RATIOS,
    "actual_document_counts": {k: len(v) for k, v in split_ids.items()},
    "actual_character_counts": {k: sum(len(x["text"]) for x in items if x["split"] == k) for k in SPLIT_RATIOS},
    "actual_character_ratios": {k: round(sum(len(x["text"]) for x in items if x["split"] == k) / sum(len(x["text"]) for x in items), 4) for k in SPLIT_RATIOS},
    "cleaning": [
        "removidos apenas os delimitadores técnicos externos do corpus mestre",
        "Unicode NFC",
        "finais de linha normalizados para LF",
        "espaços finais de linha removidos",
        "sequências de mais de duas linhas vazias reduzidas a duas",
        "cabeçalhos, notas editoriais, índices e conteúdo literário preservados",
    ],
    "exact_duplicate_groups": len(duplicate_groups),
    "leakage_note": "Validação/teste são arquivos não vistos. Fingerprint impede duplicatas integrais após normalização de espaços, mas não prova ausência de passagens repetidas entre obras distintas.",
    "files": ["train.jsonl", "validation.jsonl", "test.jsonl", "auditoria_documentos.csv", "auditoria_particoes.csv"],
}
(PREP_DIR / "manifesto_modelagem.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Arquivos gerados:")
for path in sorted(PREP_DIR.iterdir()):
    print(path.name, path.stat().st_size, "bytes")


Arquivos gerados:
auditoria_documentos.csv 42425 bytes
auditoria_particoes.csv 395 bytes
manifesto_modelagem.json 1588 bytes
test.jsonl 1325011 bytes
train.jsonl 11821395 bytes
validation.jsonl 1427693 bytes


## 5. Verificações finais

In [6]:
# Reabre o JSONL e confirma codificação, estrutura, cobertura e split disjunto.
loaded = {}
for split in SPLIT_RATIOS:
    path = PREP_DIR / f"{split}.jsonl"
    data = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]
    loaded[split] = data
    assert all(isinstance(row["text"], str) and row["text"] for row in data)
    assert all("\ufffd" not in row["text"] for row in data)
    assert all("===== INÍCIO DO ITEM:" not in row["text"] for row in data)
    assert all("===== FIM DO ITEM:" not in row["text"] for row in data)

loaded_ids = {s: {r["document_id"] for r in rows} for s, rows in loaded.items()}
assert set.union(*loaded_ids.values()) == set(ids)
assert sum(map(len, loaded_ids.values())) == len(ids)
assert not (loaded_ids["train"] & loaded_ids["validation"])
assert not (loaded_ids["train"] & loaded_ids["test"])
assert not (loaded_ids["validation"] & loaded_ids["test"])
print("Verificações passaram: JSONL válido, cobertura completa, sem wrapper residual e sem documento em mais de um split.")
print("Observação: o processamento é conservador; a próxima etapa deve tokenizar sem concatenar artificialmente obras distintas.")


Verificações passaram: JSONL válido, cobertura completa, sem wrapper residual e sem documento em mais de um split.
Observação: o processamento é conservador; a próxima etapa deve tokenizar sem concatenar artificialmente obras distintas.
